# Session 2 | Transformer 架构（3.5h）

**学习目标**：Self-Attention → Transformer Block → GPT 组装 → 文本生成

| 时间 | 内容 | 类型 |
|------|------|------|
| 0:00-0:10 | 回顾：静态向量的局限 | 讲解 |
| 0:10-0:50 | Self-Attention：QKV 机制 | 代码随讲 + 练习 |
| 0:50-1:15 | Multi-Head Attention | 代码随讲 + 练习 |
| 1:15-1:25 | 休息 | |
| 1:25-2:00 | Transformer Block | 代码随讲 + 练习 |
| 2:00-2:40 | GPT 组装与生成 | 代码随讲 + 练习 |
| 2:40-3:10 | Mini-project + 练习5 | 实操 |
| 3:10-3:30 | 架构对比 + 总结 | 讲解 |

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), "utils"))
sys.path.insert(0, "utils")

In [ ]:
# ── 中文字体配置（确保图表中文不乱码）──
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager as fm

_font_path = "../fonts/NotoSansCJKsc-Regular.otf"
if not os.path.exists(_font_path):
    _font_path = "../assets/fonts/NotoSansCJKsc-Regular.otf"
if os.path.exists(_font_path):
    fm.fontManager.addfont(_font_path)
    _cjk = fm.FontProperties(fname=_font_path).get_name()
    mpl.rcParams["font.sans-serif"] = [_cjk] + [
        f for f in mpl.rcParams["font.sans-serif"] if f != _cjk
    ]
    print(f"已加载中文字体: {_cjk}")
else:
    print("未找到自定义中文字体，使用系统字体")
    mpl.rcParams["font.sans-serif"] = [
        "Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans",
    ]
mpl.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 120

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import seaborn as sns

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")

---

# Part 1 | 回顾：为什么静态向量不够？

上午我们学了 Word Embedding（词嵌入），把每个词映射到一个固定向量。

**问题**：同一个词在不同语境中含义不同，但静态 Embedding 给出的向量完全一样：

- "我去**银行**存钱" → `bank` = 金融机构
- "河**边**有一排柳树" → `bank` = 河岸

**我们需要**：根据上下文动态调整每个词的表示 → 这就是 **Self-Attention（自注意力）** 的核心思想。

> 核心直觉：让每个词"看看"句子里的其他词，决定该关注谁，然后用关注到的信息更新自己的表示。

**直觉理解 QKV：** 想象你在图书馆找书。Query（查询）是你脑中的问题，Key（键）是每本书封面上的标签，Value（值）是书里的实际内容。Attention 的本质就是：用你的问题去匹配所有标签，找到最相关的书，然后提取其中的信息。

换句话说，每个词都会"提出问题"（Query），同时也会"亮出自己的标签"（Key）和"准备好自己的内容"（Value）。通过 Query 和所有 Key 的匹配程度，模型决定该从哪些词那里获取多少信息。

---

# Part 2 | Self-Attention：QKV 机制

## 核心公式

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

**三个角色**：
- **Q (Query)** — 我在找什么信息？
- **K (Key)** — 我有什么信息可以提供？
- **V (Value)** — 我实际提供的内容

**类比**：图书馆检索
- Q = 你的搜索词
- K = 每本书的标签
- V = 书的实际内容
- 搜索词和标签越匹配，你越关注那本书的内容

**为什么要除以 $\sqrt{d_k}$？**
当维度很大时，点积值会很大 → softmax 输出接近 one-hot → 梯度消失。缩放让分布更平滑。

### Step-by-step：用小矩阵理解 Self-Attention

In [ ]:
# 用一个 3 词、4 维的迷你例子
# 假设 3 个词："猫" "坐" "垫"
torch.manual_seed(42)

seq_len, d_model = 3, 4
x = torch.randn(seq_len, d_model)
print("输入 x (每行一个词的 embedding):")
print(x)
print(f"\n形状: [{seq_len}, {d_model}] = [词数, 维度]")

In [ ]:
# Step 1: 线性投影得到 Q, K, V
d_k = 4
W_q = nn.Linear(d_model, d_k, bias=False)
W_k = nn.Linear(d_model, d_k, bias=False)
W_v = nn.Linear(d_model, d_k, bias=False)

Q = W_q(x)  # [3, 4]
K = W_k(x)  # [3, 4]
V = W_v(x)  # [3, 4]

print("Q (Query):")
print(Q.data)
print("\nK (Key):")
print(K.data)
print("\nV (Value):")
print(V.data)

In [ ]:
# Step 2: 计算注意力分数 scores = Q @ K^T / sqrt(d_k)
scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)
print(f"注意力分数 (缩放后) [{seq_len}x{seq_len}]:")
print(scores.data)
print("\n解读: scores[i][j] = 词i对词j的关注程度")

In [ ]:
# Step 3: Softmax 得到注意力权重
attention_weights = F.softmax(scores, dim=-1)
print("注意力权重 (每行求和=1):")
print(attention_weights.data)
print(f"\n第一行求和验证: {attention_weights[0].sum().item():.4f}")

In [ ]:
# Step 4: 加权求和得到输出
output = torch.matmul(attention_weights, V)
print(f"输出形状: {output.shape}  (和输入一样!)")
print("\n输出 (每个词的新表示 = 融合了上下文信息):")
print(output.data)

In [ ]:
# 可视化注意力权重
words = ["猫", "坐", "垫"]

plt.figure(figsize=(5, 4))
sns.heatmap(attention_weights.detach().numpy(), annot=True, fmt='.3f',
            cmap='Blues', xticklabels=words, yticklabels=words)
plt.xlabel('Key (被关注的词)')
plt.ylabel('Query (发起关注的词)')
plt.title('Self-Attention 权重热力图')
plt.tight_layout()
plt.show()

### 完整的 Self-Attention 函数

In [ ]:
def self_attention(x, d_k):
    """
    最简单的 Self-Attention 实现 (Q=K=V=x, 无投影矩阵)
    
    x: [batch, seq_len, d_model]
    d_k: 注意力维度
    返回: (output, attention_weights)
    """
    Q, K, V = x, x, x
    
    # 1. 计算注意力分数
    scores = torch.matmul(Q, K.transpose(-2, -1))  # [batch, seq_len, seq_len]
    
    # 2. 缩放
    scores = scores / np.sqrt(d_k)
    
    # 3. Softmax
    attention_weights = F.softmax(scores, dim=-1)
    
    # 4. 加权求和
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

# 测试
batch_size, seq_len, d_model = 1, 5, 8
x = torch.randn(batch_size, seq_len, d_model)
output, weights = self_attention(x, d_k=d_model)

print(f"输入形状: {x.shape}")
print(f"输出形状: {output.shape}")
print(f"注意力权重形状: {weights.shape}")

🔧 **进度：** 我们已经理解了注意力的数学原理，现在来把它实现为一个可复用的 PyTorch 模块。

### 带投影矩阵的 SelfAttention 类

In [ ]:
class SelfAttention(nn.Module):
    """标准 Self-Attention 层"""
    def __init__(self, d_model, d_k=None):
        super().__init__()
        self.d_k = d_k or d_model
        
        # Q, K, V 投影矩阵
        self.W_q = nn.Linear(d_model, self.d_k, bias=False)
        self.W_k = nn.Linear(d_model, self.d_k, bias=False)
        self.W_v = nn.Linear(d_model, self.d_k, bias=False)
        self.W_o = nn.Linear(self.d_k, d_model, bias=False)  # 输出投影
    
    def forward(self, x, mask=None):
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attention_weights = F.softmax(scores, dim=-1)
        context = torch.matmul(attention_weights, V)
        output = self.W_o(context)
        
        return output, attention_weights

# 测试
attention = SelfAttention(d_model=32, d_k=16)
x = torch.randn(2, 10, 32)
output, weights = attention(x)
print(f"输入: {x.shape}")
print(f"输出: {output.shape}")
print(f"参数量: {sum(p.numel() for p in attention.parameters())}")

### 因果掩码（Causal Mask）

在 GPT 这样的生成模型中，预测第 t 个词时只能看到前 t-1 个词，不能"偷看"未来。

用一个下三角矩阵实现：上三角部分设为 `-inf`，softmax 后变为 0。

In [ ]:
# 可视化因果掩码
seq_len_mask = 6
mask = torch.tril(torch.ones(seq_len_mask, seq_len_mask))

plt.figure(figsize=(5, 4))
sns.heatmap(mask.numpy(), annot=True, fmt='.0f', cmap='Greens',
            xticklabels=[f't={i}' for i in range(seq_len_mask)],
            yticklabels=[f't={i}' for i in range(seq_len_mask)])
plt.xlabel('Key (被关注的位置)')
plt.ylabel('Query (当前位置)')
plt.title('因果掩码 (Causal Mask)')
plt.tight_layout()
plt.show()

print("1 = 可以关注, 0 = 不可关注 (被遮蔽)")
print("每个位置只能看到自己和之前的词")

---

## 🏋️ 练习 1：手算 Attention 分数（10 min）⭐ 入门

**目标**：用简化的数值手算 Self-Attention 全过程，加深对公式的理解。

**设定**：2 个词，每个词 64 维，所有维度值相同。简化为 Q=K=V=x（无投影矩阵）。

给定：
- `x1 = [1.323, 1.323, ..., 1.323]` （64 维）
- `x2 = [1.134, 1.134, ..., 1.134]` （64 维）

**步骤**：
1. 计算 `q2·k1` 和 `q2·k2`（点积）
2. 除以 `sqrt(d_k)` 缩放
3. Softmax 得到权重
4. 加权求和得到 `z2`

**预期输出**：
```
Step 1: q2·k1 = 96.27, q2·k2 = 82.31
Step 2: scaled scores = [12.03, 10.29]
Step 3: attention weights = [0.851, 0.149]
Step 4: z2 = 0.851 * 1.323 + 0.149 * 1.134 = 1.2949
✅ 所有验证通过！
```

In [ ]:
# 练习 1：手算并验证 q2 的注意力输出
import numpy as np
import torch
import torch.nn.functional as F

d_k = 64
x1_val = 1.323
x2_val = 1.134

# Step 1: 计算点积 q2·k1 和 q2·k2
# ↓↓↓ 你的代码 ↓↓↓
q2_k1 = d_k * x2_val * x1_val
q2_k2 = d_k * x2_val * x2_val
# ↑↑↑ 你的代码 ↑↑↑

print(f"Step 1: q2·k1 = {q2_k1:.2f}, q2·k2 = {q2_k2:.2f}")

# Step 2: 缩放（除以 sqrt(d_k)）
# ↓↓↓ 你的代码 ↓↓↓
sqrt_dk = np.sqrt(d_k)
scaled_q2_k1 = q2_k1 / sqrt_dk
scaled_q2_k2 = q2_k2 / sqrt_dk
# ↑↑↑ 你的代码 ↑↑↑

print(f"Step 2: scaled scores = [{scaled_q2_k1:.2f}, {scaled_q2_k2:.2f}]")

# Step 3: Softmax 得到注意力权重
# ↓↓↓ 你的代码 ↓↓↓
scores = torch.tensor([scaled_q2_k1, scaled_q2_k2])
weights = F.softmax(scores, dim=-1)
# ↑↑↑ 你的代码 ↑↑↑

print(f"Step 3: attention weights = [{weights[0]:.3f}, {weights[1]:.3f}]")

# Step 4: 加权求和 z2 = w1 * v1 + w2 * v2
# ↓↓↓ 你的代码 ↓↓↓
z2_val = weights[0].item() * x1_val + weights[1].item() * x2_val
# ↑↑↑ 你的代码 ↑↑↑

print(f"Step 4: z2 = {weights[0]:.3f} * {x1_val} + {weights[1]:.3f} * {x2_val} = {z2_val:.4f}")

# 验证
assert abs(q2_k1 - 96.27) < 0.5, f"q2_k1 应约等于 96.27，得到 {q2_k1:.2f}"
assert abs(q2_k2 - 82.31) < 0.5, f"q2_k2 应约等于 82.31，得到 {q2_k2:.2f}"
assert abs(scaled_q2_k1 - 12.03) < 0.1, f"缩放后应约等于 12.03，得到 {scaled_q2_k1:.2f}"
assert abs(z2_val - 1.295) < 0.05, f"z2 应约等于 1.295，得到 {z2_val:.4f}"
print("\n✅ 所有验证通过！")

<details>
<summary>💡 提示1：点积计算</summary>

所有维度值相同时，点积 = 维数 × 值1 × 值2：

```python
q2_k1 = d_k * x2_val * x1_val
q2_k2 = d_k * x2_val * x2_val
```

</details>

<details>
<summary>💡 提示2：缩放</summary>

```python
sqrt_dk = np.sqrt(d_k)  # = 8.0
scaled_q2_k1 = q2_k1 / sqrt_dk
scaled_q2_k2 = q2_k2 / sqrt_dk
```

</details>

<details>
<summary>💡 提示3：加权求和</summary>

```python
z2_val = weights[0].item() * x1_val + weights[1].item() * x2_val
```

</details>

---

# Part 3 | Multi-Head Attention（多头注意力）

**核心思想**：一个 attention head 只能学一种关系模式。多个 head 并行，能同时捕获不同类型的关系：
- Head 1 可能关注语法关系（主语-谓语）
- Head 2 可能关注语义关系（同义词）
- Head 3 可能关注位置关系（相邻词）

**实现方式**：把 `d_model` 拆成 `n_heads` 份，每份独立做 attention，最后拼回来。

In [ ]:
class MultiHeadAttention(nn.Module):
    """多头注意力：多个 Attention 并行，然后拼接"""
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, x, mask=None):
        batch_size, seq_len, d_model = x.shape
        
        # 1. 线性投影
        Q = self.W_q(x)  # [batch, seq_len, d_model]
        K = self.W_k(x)
        V = self.W_v(x)
        
        # 2. 分割成多头
        # [batch, seq_len, d_model] -> [batch, n_heads, seq_len, d_k]
        Q = Q.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        
        # 3. 计算注意力
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attention_weights = F.softmax(scores, dim=-1)
        context = torch.matmul(attention_weights, V)
        
        # 4. 拼接多头
        # [batch, n_heads, seq_len, d_k] -> [batch, seq_len, d_model]
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        
        # 5. 输出投影
        output = self.W_o(context)
        
        return output, attention_weights

# 测试
mha = MultiHeadAttention(d_model=64, n_heads=8)
x = torch.randn(2, 10, 64)
output, weights = mha(x)

print(f"输入: {x.shape}")
print(f"输出: {output.shape}")
print(f"注意力权重: {weights.shape}  (8个头，每个头10x10)")

In [ ]:
# 可视化不同 head 的注意力模式
torch.manual_seed(42)
mha_vis = MultiHeadAttention(d_model=64, n_heads=4)
x_vis = torch.randn(1, 6, 64)
_, weights_vis = mha_vis(x_vis)

words_vis = ["我", "喜欢", "用", "大", "模型", "写代码"]

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for h in range(4):
    w = weights_vis[0, h].detach().numpy()
    sns.heatmap(w, ax=axes[h], cmap='Blues', annot=True, fmt='.2f',
                xticklabels=words_vis, yticklabels=words_vis, cbar=False)
    axes[h].set_title(f'Head {h}')
plt.suptitle('不同 Head 关注不同的关系模式', fontsize=13)
plt.tight_layout()
plt.show()

---

## 🏋️ 练习 2：注意力模式侦探（15 min）⭐⭐ 进阶

**目标**：量化分析不同 head 的关注模式，理解多头注意力的分工。

**步骤**：
1. 创建 `MultiHeadAttention`，对两个不同输入跑前向传播
2. 实现注意力熵计算：`H = -sum(w * log(w + eps))`
   - 高熵 = 广泛关注（均匀分布）
   - 低熵 = 集中关注（尖锐分布）
3. 打印分析表格，找出最集中的 head

**预期输出**：
```
Head | 输入A 熵 | 输入B 熵 | 差异
-----|---------|---------|------
   0 |  1.xxxx |  1.xxxx | 0.xxxx
   1 |  0.xxxx |  1.xxxx | 0.xxxx
  ...
🔍 最集中的Head: X（熵=X.XXXX）
✅ 所有验证通过！
```

In [ ]:
# 练习 2：注意力模式侦探
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

d_model = 32
n_heads = 4
seq_len = 6

mha = MultiHeadAttention(d_model, n_heads)

# 两个不同分布的输入
torch.manual_seed(42)
input_a = torch.randn(1, seq_len, d_model)
input_b = torch.randn(1, seq_len, d_model) * 2 + 1

# 步骤1：提取每个 head 的注意力权重
# ↓↓↓ 你的代码 ↓↓↓
d_k = d_model // n_heads

def get_attention_weights(mha, x):
    """提取每个 head 的注意力权重矩阵"""
    B, T, C = x.shape
    Q = mha.W_q(x).view(B, T, n_heads, d_k).transpose(1, 2)
    K = mha.W_k(x).view(B, T, n_heads, d_k).transpose(1, 2)
    scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)
    weights = F.softmax(scores, dim=-1)
    return weights.squeeze(0)  # (n_heads, T, T)
# ↑↑↑ 你的代码 ↑↑↑

with torch.no_grad():
    weights_a = get_attention_weights(mha, input_a)
    weights_b = get_attention_weights(mha, input_b)

# 步骤2：实现注意力熵计算
# ↓↓↓ 你的代码 ↓↓↓
def attention_entropy(weights):
    """计算注意力分布的平均熵"""
    eps = 1e-8
    H = -(weights * torch.log(weights + eps)).sum(dim=-1).mean()
    return H.item()
# ↑↑↑ 你的代码 ↑↑↑

# 步骤3：打印分析表格
print(f"{'Head':>4} | {'输入A 熵':>8} | {'输入B 熵':>8} | {'差异':>6}")
print("-" * 40)
for h in range(n_heads):
    h_a = attention_entropy(weights_a[h])
    h_b = attention_entropy(weights_b[h])
    print(f"{h:>4} | {h_a:>8.4f} | {h_b:>8.4f} | {abs(h_a - h_b):>6.4f}")

entropies_a = [attention_entropy(weights_a[h]) for h in range(n_heads)]
min_head = int(np.argmin(entropies_a))
print(f"\n🔍 最集中的Head: {min_head}（熵={entropies_a[min_head]:.4f}）")

# 验证
assert weights_a.shape == (n_heads, seq_len, seq_len), f"权重形状应为 ({n_heads}, {seq_len}, {seq_len})"
h_test = attention_entropy(weights_a[0])
assert 0 < h_test < 5, f"熵值 {h_test} 不在合理范围内"
print("\n✅ 所有验证通过！")

<details>
<summary>💡 提示1：如何获取 Q、K</summary>

利用 `mha` 的权重矩阵做投影，再 reshape 成多头格式：

```python
Q = mha.W_q(x).view(B, T, n_heads, d_k).transpose(1, 2)
K = mha.W_k(x).view(B, T, n_heads, d_k).transpose(1, 2)
```

</details>

<details>
<summary>💡 提示2：熵的计算公式</summary>

```python
H = -(weights * torch.log(weights + eps)).sum(dim=-1).mean()
```

对最后一个维度求和得到每行的熵，再取平均。记得用 `.item()` 转为标量。

</details>

<details>
<summary>💡 提示3：完整的 get_attention_weights</summary>

```python
Q = mha.W_q(x).view(B, T, n_heads, d_k).transpose(1, 2)
K = mha.W_k(x).view(B, T, n_heads, d_k).transpose(1, 2)
scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)
weights = F.softmax(scores, dim=-1)
return weights.squeeze(0)
```

</details>

---

# *休息 10 分钟*

---

# Part 4 | Transformer Block：构建 LLM 的积木

一个 Transformer Block 由以下组件构成：

```
输入 x
  │
  ├──→ LayerNorm → Multi-Head Attention ──→ (+) ← 残差连接
  │                                          │
  │                                          ├──→ LayerNorm → FFN ──→ (+) ← 残差连接
  │                                          │                         │
  │                                          └─────────────────────────┘
  └──────────────────────────────────────────→ 输出
```

**三个关键组件**：
1. **残差连接（Residual Connection）** — 解决深层网络梯度消失
2. **层归一化（Layer Normalization）** — 稳定训练
3. **前馈网络（Feed-Forward Network, FFN）** — 增加非线性变换能力

## 4.1 残差连接（Residual Connection）

核心思想很简单：`output = f(x) + x`

**为什么重要？** 没有残差连接，10层以上的网络梯度会迅速消失/爆炸。

In [ ]:
# 对比有无残差连接的梯度表现
d_model = 64

class ResidualBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.linear = nn.Linear(d_model, d_model)
    def forward(self, x):
        return self.linear(x) + x  # 关键: + x

# 无残差: 10 层普通线性层
x1 = torch.randn(1, 10, d_model, requires_grad=True)
layers_no_res = nn.Sequential(*[nn.Linear(d_model, d_model) for _ in range(10)])
out1 = layers_no_res(x1)
out1.sum().backward()
print(f"无残差 - 输出范围: [{out1.min().item():.2f}, {out1.max().item():.2f}]")

# 有残差: 10 层残差块
x2 = torch.randn(1, 10, d_model, requires_grad=True)
layers_with_res = nn.ModuleList([ResidualBlock(d_model) for _ in range(10)])
out2 = x2.clone()
for layer in layers_with_res:
    out2 = layer(out2)
out2.sum().backward()
print(f"有残差 - 输出范围: [{out2.min().item():.2f}, {out2.max().item():.2f}]")
print("\n残差连接让信号更稳定！")

## 4.2 层归一化（Layer Normalization）

对每个样本的特征维度做归一化（均值=0，方差=1），再做仿射变换。

$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

In [ ]:
class LayerNorm(nn.Module):
    """手写 LayerNorm"""
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
    
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * x_norm + self.beta

# 测试
x = torch.randn(2, 5, 64) * 10 + 5  # 均值不为0，方差不为1
layer_norm = LayerNorm(64)
x_normed = layer_norm(x)

print("归一化前:")
print(f"  均值: {x.mean(dim=-1)[0].data}")
print(f"  方差: {x.var(dim=-1)[0].data}")
print("\n归一化后:")
print(f"  均值: {x_normed.mean(dim=-1)[0].data}")  # 接近 0
print(f"  方差: {x_normed.var(dim=-1)[0].data}")   # 接近 1

# 和 PyTorch 官方对比
pytorch_ln = nn.LayerNorm(64)
x_pytorch = pytorch_ln(x)
print(f"\n与 PyTorch 官方实现的差距: {(x_normed - x_pytorch).abs().max().item():.8f}")

🔧 **进度：** 注意力层已搭建完成！现在构建前馈网络层。

## 4.3 前馈网络（Feed-Forward Network）

两层线性变换 + 激活函数。先扩展维度（4x），再压缩回来。现代 LLM 通常用 GELU 激活。

In [ ]:
class FeedForward(nn.Module):
    """Transformer 中的 FFN"""
    def __init__(self, d_model, d_ff=None, dropout=0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model  # 默认 4 倍扩展
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        x = self.linear1(x)
        x = F.gelu(x)  # 现代 LLM 用 GELU
        x = self.dropout(x)
        x = self.linear2(x)
        return x

# 测试
ffn = FeedForward(d_model=64, d_ff=256)
x = torch.randn(2, 10, 64)
out = ffn(x)
print(f"输入: {x.shape}")
print(f"中间层: [2, 10, 256]  (4x 扩展)")
print(f"输出: {out.shape}")
print(f"参数量: {sum(p.numel() for p in ffn.parameters()):,}")

🔧 **进度：** 注意力层 ✓ 前馈层 ✓ 现在把它们组装成一个完整的 Transformer Block。

## 4.4 组装 Transformer Block

In [ ]:
class MultiHeadAttentionV2(nn.Module):
    """支持 dropout 的 MultiHeadAttention (用于 TransformerBlock)"""
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        B, T, C = x.shape
        Q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn = self.dropout(F.softmax(scores, dim=-1))
        context = torch.matmul(attn, V)
        context = context.transpose(1, 2).contiguous().view(B, T, C)
        return self.W_o(context)

In [ ]:
class TransformerBlock(nn.Module):
    """
    完整的 Transformer Block (Pre-LN 版本)
    
    结构:
    x -> LayerNorm -> Attention -> (+) -> LayerNorm -> FFN -> (+)
         |__________________________|    |___________________|  
                残差连接                       残差连接
    """
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.attention = MultiHeadAttentionV2(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Attention + 残差
        x = x + self.dropout(self.attention(self.ln1(x), mask))
        # FFN + 残差
        x = x + self.dropout(self.ffn(self.ln2(x)))
        return x

# 测试
block = TransformerBlock(d_model=64, n_heads=8, d_ff=256)
x = torch.randn(2, 10, 64)
out = block(x)

print("Transformer Block:")
print(f"  输入: {x.shape}")
print(f"  输出: {out.shape}")
print(f"  参数量: {sum(p.numel() for p in block.parameters()):,}")

---

## 🏋️ 练习 3：从组件拼装 TransformerBlock（10 min）⭐⭐ 进阶

**目标**：用已有组件搭建完整的 Transformer Block，理解各部分如何连接。

**步骤**：
1. 在 `__init__` 中创建 5 个组件：`ln1`, `ln2`, `attention`, `ffn`, `dropout`
2. 在 `forward` 中按 Pre-LN 结构连接：`x → LN1 → Attention → Dropout → (+x) → LN2 → FFN → Dropout → (+x)`

**需要用到**：`nn.LayerNorm`, `MultiHeadAttentionV2`, `FeedForward`, `nn.Dropout`

**预期输出**：
```
输入: torch.Size([2, 10, 64])
输出: torch.Size([2, 10, 64])
参数量: xx,xxx
✅ 通过!
```

In [ ]:
# 练习 3：从组件拼装 TransformerBlock
class MyTransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        # ↓↓↓ 你的代码（约5行）↓↓↓
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.attention = MultiHeadAttentionV2(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)
        # ↑↑↑ 你的代码 ↑↑↑

    def forward(self, x, mask=None):
        # ↓↓↓ 你的代码（约2行）↓↓↓
        x = x + self.dropout(self.attention(self.ln1(x), mask))
        x = x + self.dropout(self.ffn(self.ln2(x)))
        # ↑↑↑ 你的代码 ↑↑↑
        return x

# 测试
my_block = MyTransformerBlock(d_model=64, n_heads=8, d_ff=256)
x_test = torch.randn(2, 10, 64)
out_test = my_block(x_test)
print(f"输入: {x_test.shape}")
print(f"输出: {out_test.shape}")

# 验证
assert out_test.shape == torch.Size([2, 10, 64]), f"输出形状应为 [2, 10, 64]，得到 {out_test.shape}"
param_count = sum(p.numel() for p in my_block.parameters())
assert param_count > 0, "模型没有参数，请检查 __init__"
print(f"参数量: {param_count:,}")
print("✅ 通过!")

<details>
<summary>💡 提示1：__init__ 中需要的组件</summary>

```python
self.ln1 = nn.LayerNorm(d_model)
self.ln2 = nn.LayerNorm(d_model)
self.attention = MultiHeadAttentionV2(d_model, n_heads, dropout)
self.ffn = FeedForward(d_model, d_ff, dropout)
self.dropout = nn.Dropout(dropout)
```

</details>

<details>
<summary>💡 提示2：forward 的连接方式</summary>

Pre-LN 结构：先归一化再做变换，残差连接在外层。

```python
x = x + self.dropout(self.attention(self.ln1(x), mask))
x = x + self.dropout(self.ffn(self.ln2(x)))
```

</details>

🔧 **进度：** 单个 Block 已就绪，现在堆叠多层构建完整的 Transformer 模型。

### 堆叠多层 Transformer

In [ ]:
class Transformer(nn.Module):
    """堆叠多个 Transformer Block"""
    def __init__(self, n_layers, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.ln_final = nn.LayerNorm(d_model)
    
    def forward(self, x, mask=None):
        for block in self.blocks:
            x = block(x, mask)
        return self.ln_final(x)

# 测试 6 层 Transformer
model = Transformer(n_layers=6, d_model=64, n_heads=8, d_ff=256)
x = torch.randn(2, 10, 64)
out = model(x)

print(f"6层 Transformer:")
print(f"  输入: {x.shape}")
print(f"  输出: {out.shape}")
print(f"  总参数量: {sum(p.numel() for p in model.parameters()):,}")

---

# Part 5 | GPT 组装：从积木到完整模型

GPT (Generative Pre-trained Transformer) 的完整结构：

```
Token IDs
    │
    ├── Token Embedding (词嵌入)
    │         +
    ├── Position Embedding (位置编码)  ← 让模型知道词的顺序
    │
    ▼
  Dropout
    │
    ├── Transformer Block × N
    │
    ▼
  LayerNorm
    │
    ▼
  Linear (lm_head)  → 输出 vocab_size 维的 logits
    │
    ▼
  Softmax → 下一个词的概率分布
```

## 5.1 位置编码（Positional Encoding）

Self-Attention 本身不知道词的顺序 — "猫吃鱼" 和 "鱼吃猫" 的 attention 分数完全一样！

**解决方法**：给每个位置加一个唯一的编码向量。

两种主流方式：
- **正弦位置编码（Sinusoidal）** — Transformer 原始论文，固定的
- **可学习位置编码（Learned）** — GPT 系列采用
- **旋转位置编码（RoPE）** — LLaMA 等现代模型采用，支持外推更长序列

> 这里我们实现正弦位置编码来理解原理，GPT 模型中用可学习位置编码。

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    """
    正弦位置编码 (Transformer 原始论文)
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)  # 偶数维度
        pe[:, 1::2] = torch.cos(position * div_term)  # 奇数维度
        
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# 可视化位置编码
pos_enc = SinusoidalPositionalEncoding(d_model=64)
pe = pos_enc.pe[0, :100, :].numpy()

plt.figure(figsize=(12, 4))
plt.imshow(pe.T, aspect='auto', cmap='RdBu_r')
plt.colorbar(label='编码值')
plt.xlabel('位置 (Position)')
plt.ylabel('维度 (Dimension)')
plt.title('正弦位置编码可视化')
plt.tight_layout()
plt.show()

print("每个位置有唯一的编码模式！")

## 5.2 GPT 完整模型

In [ ]:
class CausalSelfAttention(nn.Module):
    """带因果掩码的自注意力"""
    def __init__(self, config):
        super().__init__()
        assert config['n_embd'] % config['n_head'] == 0
        
        self.n_head = config['n_head']
        self.n_embd = config['n_embd']
        self.head_dim = config['n_embd'] // config['n_head']
        
        self.c_attn = nn.Linear(config['n_embd'], 3 * config['n_embd'])
        self.c_proj = nn.Linear(config['n_embd'], config['n_embd'])
        self.dropout = nn.Dropout(config['dropout'])
        
        # 因果掩码
        self.register_buffer("mask", torch.tril(
            torch.ones(config['block_size'], config['block_size'])
        ).view(1, 1, config['block_size'], config['block_size']))
    
    def forward(self, x):
        B, T, C = x.size()
        
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        
        att = (q @ k.transpose(-2, -1)) / np.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.dropout(att)
        
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config['n_embd'], 4 * config['n_embd'])
        self.c_proj = nn.Linear(4 * config['n_embd'], config['n_embd'])
        self.dropout = nn.Dropout(config['dropout'])
    
    def forward(self, x):
        x = F.gelu(self.c_fc(x))
        return self.dropout(self.c_proj(x))


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config['n_embd'])
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config['n_embd'])
        self.mlp = MLP(config)
    
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

In [ ]:
class GPT(nn.Module):
    """GPT 语言模型"""
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config['vocab_size'], config['n_embd']),   # token embedding
            wpe = nn.Embedding(config['block_size'], config['n_embd']),   # position embedding (可学习)
            drop = nn.Dropout(config['dropout']),
            h = nn.ModuleList([Block(config) for _ in range(config['n_layer'])]),
            ln_f = nn.LayerNorm(config['n_embd']),
        ))
        self.lm_head = nn.Linear(config['n_embd'], config['vocab_size'], bias=False)
        
        # 权重共享: embedding 和 lm_head 共享权重 (减少参数量)
        self.transformer.wte.weight = self.lm_head.weight
        
        self.apply(self._init_weights)
        print(f"GPT 参数量: {self.get_num_params():,}")
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def get_num_params(self):
        return sum(p.numel() for p in self.parameters())
    
    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.config['block_size']
        
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        tok_emb = self.transformer.wte(idx)    # [B, T, n_embd]
        pos_emb = self.transformer.wpe(pos)    # [T, n_embd]
        x = self.transformer.drop(tok_emb + pos_emb)
        
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        
        logits = self.lm_head(x)  # [B, T, vocab_size]
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        
        return logits, loss

# 创建一个小型 GPT
config = {
    'vocab_size': 1000,
    'block_size': 128,
    'n_layer': 4,
    'n_head': 4,
    'n_embd': 128,
    'dropout': 0.1,
}

gpt = GPT(config)

In [ ]:
# 测试前向传播
idx = torch.randint(0, config['vocab_size'], (2, 20))  # 2个样本，20个token
logits, _ = gpt(idx)
print(f"输入: {idx.shape}  (batch=2, seq_len=20)")
print(f"输出 logits: {logits.shape}  (batch=2, seq_len=20, vocab={config['vocab_size']})")

# 带目标的前向传播 (计算 loss)
targets = torch.randint(0, config['vocab_size'], (2, 20))
logits, loss = gpt(idx, targets)
print(f"\nLoss: {loss.item():.4f}  (随机初始化，期望 ≈ ln(1000) ≈ {np.log(1000):.2f})")

## 5.3 文本生成：采样策略

In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None):
    """
    自回归文本生成
    
    idx: [B, T] 初始 token 序列
    temperature: 控制随机性 (越高越随机)
    top_k: 只从概率最高的 k 个中采样
    top_p: 只从累积概率达到 p 的 token 中采样 (nucleus sampling)
    """
    model.eval()
    
    for _ in range(max_new_tokens):
        # 截断到最大长度
        idx_cond = idx if idx.size(1) <= model.config['block_size'] else idx[:, -model.config['block_size']:]
        
        # 前向传播
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]  # 只取最后一个位置 [B, vocab_size]
        
        # Temperature
        logits = logits / temperature
        
        # Top-k 过滤
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')
        
        # Top-p (nucleus) 过滤
        if top_p is not None:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0
            indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
            logits[indices_to_remove] = float('-inf')
        
        # 采样
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
    
    return idx

print("生成函数定义完成！")

In [ ]:
# 演示不同采样策略
gpt.eval()
idx = torch.randint(0, config['vocab_size'], (1, 5))  # 5 个起始 token

print("注意：模型未经训练，输出为随机 token。重点观察不同策略的差异。")
print(f"起始序列: {idx[0].tolist()}\n")

gen_greedy = generate(gpt, idx.clone(), max_new_tokens=10, temperature=0.1)
gen_random = generate(gpt, idx.clone(), max_new_tokens=10, temperature=1.5)
gen_topk = generate(gpt, idx.clone(), max_new_tokens=10, temperature=1.0, top_k=5)
gen_topp = generate(gpt, idx.clone(), max_new_tokens=10, temperature=1.0, top_p=0.9)

print(f"Greedy  (temp=0.1): {gen_greedy[0].tolist()}")
print(f"Random  (temp=1.5): {gen_random[0].tolist()}")
print(f"Top-k   (k=5):     {gen_topk[0].tolist()}")
print(f"Top-p   (p=0.9):   {gen_topp[0].tolist()}")

print("\n低 temperature → 更确定 (重复)")
print("高 temperature → 更随机 (多样)")
print("Top-k/Top-p → 在多样性和质量之间平衡")

---

## 🏋️ 练习 4：自己实现 Top-k 采样（15 min）⭐⭐⭐ 挑战

**目标**：从零实现采样的核心逻辑，理解 temperature 和 top-k 如何影响生成。

**步骤**：
1. 实现 `my_temperature_scale(logits, temperature)`：用 temperature 缩放 logits
2. 实现 `my_top_k_filter(logits, k)`：只保留 top-k 个 logit，其余设为 `-inf`
3. 对比三种配置的概率分布，用柱状图可视化

**预期输出**：三组柱状图显示不同 `(temperature, k)` 下的概率分布差异

In [ ]:
# 练习 4：自己实现 Top-k 采样
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# 模拟词汇表的 logits
logits = torch.tensor([2.0, 1.0, 0.5, 0.1, -1.0, -2.0])
vocab = ["好", "的", "很", "不", "了", "吗"]

# 步骤1：实现 Temperature 缩放
# ↓↓↓ 你的代码 ↓↓↓
def my_temperature_scale(logits, temperature):
    """temperature < 1 更确定，temperature > 1 更随机"""
    return logits / temperature
# ↑↑↑ 你的代码 ↑↑↑

# 步骤2：实现 Top-k 过滤
# ↓↓↓ 你的代码 ↓↓↓
def my_top_k_filter(logits, k):
    """只保留概率最高的 k 个 token，其余设为 -inf"""
    top_values, _ = torch.topk(logits, k)
    threshold = top_values[-1]
    logits[logits < threshold] = float('-inf')
    return logits
# ↑↑↑ 你的代码 ↑↑↑

# 步骤3：对比三种配置
configs = [
    {"name": "保守 (temp=0.5, k=3)", "temp": 0.5, "k": 3},
    {"name": "随机 (temp=2.0, k=6)", "temp": 2.0, "k": 6},
    {"name": "平衡 (temp=1.0, k=2)", "temp": 1.0, "k": 2},
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, cfg in enumerate(configs):
    scaled = my_temperature_scale(logits.clone(), cfg["temp"])
    filtered = my_top_k_filter(scaled, cfg["k"])
    probs = F.softmax(filtered, dim=-1)
    axes[i].bar(vocab, probs.numpy())
    axes[i].set_title(cfg["name"])
    axes[i].set_ylim(0, 1)
    axes[i].set_ylabel("概率")
plt.tight_layout()
plt.show()

# 验证
test_logits = torch.tensor([3.0, 1.0, 2.0, 0.5, -1.0])
scaled = my_temperature_scale(test_logits, 2.0)
assert torch.allclose(scaled, test_logits / 2.0), "Temperature 缩放不正确"

filtered = my_top_k_filter(torch.tensor([3.0, 1.0, 2.0, 0.5, -1.0]), 2)
assert (filtered == float('-inf')).sum() == 3, "Top-2 应过滤掉 3 个值"
print("✅ 所有验证通过！")

<details>
<summary>💡 提示1：Temperature 缩放</summary>

就是简单的除法：

```python
return logits / temperature
```

</details>

<details>
<summary>💡 提示2：Top-k 过滤思路</summary>

`torch.topk(logits, k)` 返回 top-k 的值和索引。用 `values[-1]`（最小的 top-k 值）作为阈值，低于阈值的设为 `-inf`。

</details>

<details>
<summary>💡 提示3：完整的 Top-k 实现</summary>

```python
top_values, _ = torch.topk(logits, k)
threshold = top_values[-1]
logits[logits < threshold] = float('-inf')
return logits
```

</details>

---

## 🏋️ 练习 5：参数量估算器（15 min）⭐⭐⭐ 挑战

**目标**：不创建模型，纯用公式计算参数量，理解每个组件的参数贡献。

**步骤**：
1. 根据公式实现 `estimate_params()` 函数
2. 与前面创建的 GPT 模型对比验证
3. 预测 LLaMA-7B 规模的参数量

**公式**：
- Token Embedding: `vocab_size * n_embd`
- Position Embedding: `block_size * n_embd`
- 每层 Attention: `4 * n_embd^2 + 4 * n_embd`（Q/K/V/O 各一个线性层）
- 每层 FFN: `8 * n_embd^2 + 5 * n_embd`（两个线性层，中间扩展 4 倍）
- 每层 LayerNorm: `4 * n_embd`（两个 LN，各有 gamma 和 beta）
- 最终 LayerNorm: `2 * n_embd`
- LM Head: 与 Token Embedding 共享权重（不额外计数）

**预期输出**：
```
各组件参数量:
  token_emb: xxx,xxx
  ...
  total: xxx,xxx
🔮 预测 LLaMA-7B 规模:
  估算参数量: x,xxx,xxx,xxx (x.xxB)
```

In [ ]:
# 练习 5：参数量估算器

# ↓↓↓ 你的代码 ↓↓↓
def estimate_params(vocab_size, n_layer, n_head, n_embd, block_size):
    """纯公式计算 GPT 参数量（不创建模型）"""
    token_emb = vocab_size * n_embd
    pos_emb = block_size * n_embd
    per_layer_attn = 4 * n_embd**2 + 4 * n_embd
    per_layer_ffn = 8 * n_embd**2 + 5 * n_embd
    per_layer_ln = 4 * n_embd
    final_ln = 2 * n_embd
    total = token_emb + pos_emb + n_layer * (per_layer_attn + per_layer_ffn + per_layer_ln) + final_ln
    return {
        "token_emb": token_emb, "pos_emb": pos_emb,
        "per_layer_attn": per_layer_attn, "per_layer_ffn": per_layer_ffn,
        "per_layer_ln": per_layer_ln, "final_ln": final_ln, "total": total
    }
# ↑↑↑ 你的代码 ↑↑↑

# 测试：与前面的小 GPT 对比
config_est = {
    "vocab_size": 1503,
    "n_layer": 4,
    "n_head": 4,
    "n_embd": 128,
    "block_size": 64,
}

estimated = estimate_params(**config_est)
print("各组件参数量:")
for k, v in estimated.items():
    print(f"  {k}: {v:,}")

# 预测大模型参数量
print("\n🔮 预测 LLaMA-7B 规模:")
llama_est = estimate_params(
    vocab_size=32000, n_layer=32, n_head=32, n_embd=4096, block_size=2048
)
print(f"  估算参数量: {llama_est['total']:,} ({llama_est['total']/1e9:.2f}B)")

<details>
<summary>💡 提示1：思路方向</summary>

Attention 有 4 个线性层（Q/K/V/O），每个是 `n_embd * n_embd + n_embd`（权重+偏置）。
FFN 有 2 个线性层，中间维度扩展 4 倍。

</details>

<details>
<summary>💡 提示2：Attention 参数量</summary>

```python
per_layer_attn = 4 * n_embd**2 + 4 * n_embd  # W_q, W_k, W_v, W_o
```

</details>

<details>
<summary>💡 提示3：完整实现</summary>

```python
token_emb = vocab_size * n_embd
pos_emb = block_size * n_embd
per_layer_attn = 4 * n_embd**2 + 4 * n_embd
per_layer_ffn = 8 * n_embd**2 + 5 * n_embd
per_layer_ln = 4 * n_embd
final_ln = 2 * n_embd
total = token_emb + pos_emb + n_layer * (per_layer_attn + per_layer_ffn + per_layer_ln) + final_ln
```

</details>

---

## 🎯 Mini-Project：企业场景选生成策略（15 min）⭐⭐ 进阶

**目标**：为不同企业场景选择最佳生成参数，理解 temperature 和 top_k 的实际应用。

**场景**：
1. **客服Bot**：回答需要准确、一致
2. **营销文案**：需要创意、多样性
3. **代码补全**：需要精确、合理

**步骤**：
1. 为每个场景选择 `temperature` 和 `top_k` 参数并写理由
2. 用好参数和故意差的参数分别生成，对比差异
3. 写 2 句结论

> 注意：模型未经训练，输出为随机 token ID，重点是理解参数选择的理由和观察分布差异。

In [ ]:
# Mini-Project：企业场景选生成策略

scenarios = {
    "客服Bot": {
        "prompt": "客户问：你们的退货政策是什么？\n回答：",
        # ↓↓↓ 你的代码 ↓↓↓
        "good_params": {"temperature": 0.3, "top_k": 5},
        "bad_params": {"temperature": 1.5, "top_k": 50},
        "理由": "客服回答需要准确一致，低temperature和小top_k确保输出稳定可靠",
        # ↑↑↑ 你的代码 ↑↑↑
    },
    "营销文案": {
        "prompt": "为一款智能手表写一句广告语：",
        # ↓↓↓ 你的代码 ↓↓↓
        "good_params": {"temperature": 1.0, "top_k": 40},
        "bad_params": {"temperature": 0.1, "top_k": 2},
        "理由": "营销文案需要创意多样性，高temperature和大top_k鼓励新颖表达",
        # ↑↑↑ 你的代码 ↑↑↑
    },
    "代码补全": {
        "prompt": "def fibonacci(n):\n    # 返回第n个斐波那契数\n",
        # ↓↓↓ 你的代码 ↓↓↓
        "good_params": {"temperature": 0.2, "top_k": 5},
        "bad_params": {"temperature": 1.5, "top_k": 50},
        "理由": "代码补全需要精确合理，低temperature和小top_k确保语法正确",
        # ↑↑↑ 你的代码 ↑↑↑
    },
}

# 生成对比（使用前面定义的 generate 函数和 gpt 模型）
model = gpt
for scene_name, scene in scenarios.items():
    print(f"\n{'='*50}")
    print(f"📌 场景: {scene_name}")
    print(f"💡 理由: {scene['理由']}")
    idx = torch.randint(0, config['vocab_size'], (1, 5))

    print(f"\n好参数 {scene['good_params']}:")
    for i in range(3):
        text = generate(model, idx.clone(), max_new_tokens=15, **scene["good_params"])
        print(f"  [{i+1}] {text[0].tolist()[:10]}...")

    print(f"\n差参数 {scene['bad_params']}:")
    for i in range(3):
        text = generate(model, idx.clone(), max_new_tokens=15, **scene["bad_params"])
        print(f"  [{i+1}] {text[0].tolist()[:10]}...")

print("\n📝 结论:")
print("1. 不同场景需要不同的生成策略：准确性场景用低temperature和小top_k，创意场景用高temperature和大top_k")
print("2. temperature控制分布尖锐程度，top_k控制候选词范围，两者配合可在质量和多样性之间找到最佳平衡")

<details>
<summary>💡 提示：参数选择参考</summary>

- **客服Bot**：低 temperature（0.3-0.5），小 top_k（5-10） -- 稳定准确
- **营销文案**：高 temperature（0.8-1.2），大 top_k（30-50） -- 有创意
- **代码补全**：低 temperature（0.2-0.4），小 top_k（5-10） -- 精确

</details>

---

# Part 6 | 架构对比：GPT vs BERT vs T5

Transformer 架构催生了三大流派，各有擅长的场景：

| 特性 | GPT (Decoder-only) | BERT (Encoder-only) | T5 (Encoder-Decoder) |
|------|-------------------|--------------------|-----------------------|
| **代表模型** | GPT-4, Claude, LLaMA | BERT, RoBERTa | T5, mT5, BART |
| **注意力方向** | 单向 (因果掩码) | 双向 (看全文) | 编码器双向 + 解码器单向 |
| **训练目标** | 预测下一个词 | 填空 (MLM) | Seq2Seq (输入→输出) |
| **擅长任务** | 文本生成、对话、推理 | 分类、相似度、NER | 翻译、摘要、问答 |
| **企业用途** | 聊天机器人、代码助手 | 文档分类、情感分析 | 多语言翻译、文本改写 |

### 怎么选？

| 你的需求 | 推荐架构 | 理由 |
|---------|---------|------|
| 生成长文本、对话 | GPT (Decoder-only) | 自回归生成最自然 |
| 文本分类、信息抽取 | BERT (Encoder-only) | 双向理解更全面 |
| 翻译、摘要等转换任务 | T5 (Encoder-Decoder) | 天然适合输入→输出 |
| 通用能力（啥都要） | GPT (大模型) | 规模足够大时能力涌现 |

### 2024-2025 趋势

- **Decoder-only 架构占主导**：GPT-4、Claude、Gemini、LLaMA 等都是 decoder-only
- 原因：足够大的 decoder-only 模型在理解和生成任务上都表现很好
- BERT 类模型仍在企业中广泛用于轻量级分类任务（成本低、速度快）
- Encoder-Decoder 在特定场景（如翻译）仍有优势

> **企业建议**：优先考虑 API 调用大模型（GPT-4、Claude）；需要私有化部署时考虑 LLaMA/Qwen 等开源模型；轻量级分类任务可用 BERT 微调。

---

# Day 1 总结

## 今天学了什么

**上午：深度学习基础与 Embedding**
- 神经网络基础：前向传播、反向传播、梯度下降
- 词嵌入（Word Embedding）：把离散的词变成连续向量
- 嵌入空间的语义关系

**下午：Transformer 架构**
- Self-Attention：让每个词关注上下文，动态更新表示
- Multi-Head Attention：多角度捕获不同关系
- Transformer Block：残差 + LayerNorm + FFN
- GPT：位置编码 + Transformer 堆叠 + 语言模型头
- 生成策略：Temperature, Top-k, Top-p

## 关键公式

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

$$\text{TransformerBlock}(x) = x + \text{FFN}(\text{LN}(x + \text{Attention}(\text{LN}(x))))$$

## Day 2 预告

- **上午**：SFT (Supervised Fine-Tuning) 与 LoRA 微调实战
  - 如何用自己的数据微调大模型
  - LoRA：高效微调，只改 1% 的参数
- **下午**：预训练与对齐优化
  - 大模型是怎么训练出来的
  - RLHF / DPO：让模型更好地对齐人类偏好